In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import io
import base64
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# this font supports the Japanese character used in the legend translation
plt.rcParams['font.family'] = 'Noto Sans CJK JP'

In [29]:
apps_df = pd.read_csv('Play Store Data.csv')

In [30]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [31]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [32]:
apps_df = apps_df.drop_duplicates()

required_cols = ['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Last Updated']
apps_df = apps_df.dropna(subset=required_cols)
print(f"Rows after removing duplicates and nulls: {len(apps_df)}")

Rows after removing duplicates and nulls: 8893


In [33]:
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')
apps_df = apps_df.dropna(subset=['Rating'])
apps_df = apps_df[apps_df['Rating'] <= 5]
print(f"Rows after Rating cleaning: {len(apps_df)}")

Rows after Rating cleaning: 8892


In [34]:
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')
apps_df = apps_df.dropna(subset=['Reviews'])
apps_df['Reviews'] = apps_df['Reviews'].astype(int)
print(f"Rows after Reviews cleaning: {len(apps_df)}")

Rows after Reviews cleaning: 8892


In [35]:
apps_df['Installs'] = apps_df['Installs'].astype(str).str.replace(',', '').str.replace('+', '')
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors='coerce')
apps_df = apps_df.dropna(subset=['Installs'])
apps_df['Installs'] = apps_df['Installs'].astype(int)
print(f"Rows after Installs cleaning: {len(apps_df)}")

Rows after Installs cleaning: 8892


In [36]:
def convert_size(size):
    size = str(size).strip()
    if size.endswith('M'):
        return float(size[:-1])
    elif size.endswith('k'):
        return float(size[:-1]) / 1024
    else:
        return np.nan

apps_df['Size'] = apps_df['Size'].apply(convert_size)
apps_df = apps_df.dropna(subset=['Size'])
print(f"Rows after Size cleaning: {len(apps_df)}")

Rows after Size cleaning: 7424


In [37]:
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df = apps_df.dropna(subset=['Last Updated'])

# month-year column used later for grouping and the x-axis
apps_df['Month_Year'] = apps_df['Last Updated'].dt.to_period('M')
print(f"Rows after date cleaning: {len(apps_df)}")

Rows after date cleaning: 7424


In [38]:
apps_df = apps_df[apps_df['Rating'] >= 4.2]
print(f"After Rating filter (>=4.2): {len(apps_df)}")

After Rating filter (>=4.2): 4647


In [39]:
# app name should not contain any digits
apps_df = apps_df[~apps_df['App'].str.contains('[0-9]', na=False)]
print(f"After App name filter (no digits): {len(apps_df)}")

After App name filter (no digits): 4094


In [40]:
# category must start with T or P
apps_df = apps_df[apps_df['Category'].str.startswith(('T', 'P'))]
print(f"After Category filter (starts with T or P): {len(apps_df)}")
print(apps_df['Category'].unique())

After Category filter (starts with T or P): 825
['PHOTOGRAPHY' 'TRAVEL_AND_LOCAL' 'TOOLS' 'PERSONALIZATION' 'PRODUCTIVITY'
 'PARENTING']


In [41]:
apps_df = apps_df[apps_df['Reviews'] > 1000]
print(f"After Reviews filter (>1000): {len(apps_df)}")

After Reviews filter (>1000): 504


In [42]:
apps_df = apps_df[(apps_df['Size'] >= 20) & (apps_df['Size'] <= 80)]
print(f"After Size filter (20 MB to 80 MB): {len(apps_df)}")

After Size filter (20 MB to 80 MB): 129


In [43]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Month_Year
2802,"Shutterfly: Free Prints, Photo Books, Cards, G...",PHOTOGRAPHY,4.6,98716,59.0,5000000,Free,0,Everyone,Photography,2018-08-01,5.13.1,5.0 and up,2018-08
2803,FreePrints – Free Photos Delivered,PHOTOGRAPHY,4.8,109500,37.0,1000000,Free,0,Everyone,Photography,2018-08-02,2.18.2,4.1 and up,2018-08
2811,"Face Filter, Selfie Editor - Sweet Camera",PHOTOGRAPHY,4.7,142634,22.0,10000000,Free,0,Everyone,Photography,2018-07-06,1.5.1,4.4 and up,2018-07
2822,Makeup Editor -Beauty Photo Editor & Selfie Ca...,PHOTOGRAPHY,4.5,3378,30.0,1000000,Free,0,Mature 17+,Photography,2018-07-25,1.7,4.1 and up,2018-07
2823,Makeup Photo Editor: Makeup Camera & Makeup Ed...,PHOTOGRAPHY,4.4,10525,25.0,1000000,Free,0,Everyone,Photography,2018-07-27,8.9.9,4.0 and up,2018-07


In [44]:
# nice display names for the categories, then translate the 3 required ones
cat_names = {
    'TOOLS': 'Tools',
    'TRAVEL_AND_LOCAL': 'Travel & Local',
    'PERSONALIZATION': 'Personalization',
    'PRODUCTIVITY': 'Productivity',
    'PARENTING': 'Parenting',
    'PHOTOGRAPHY': 'Photography'
}

translate = {
    'Travel & Local': 'Voyage et Local',
    'Productivity': 'Productividad',
    'Photography': '写真'
}

legend_names = {}
for cat in cat_names:
    name = cat_names[cat]
    if name in translate:
        name = translate[name]
    legend_names[cat] = name

legend_names

{'TOOLS': 'Tools',
 'TRAVEL_AND_LOCAL': 'Voyage et Local',
 'PERSONALIZATION': 'Personalization',
 'PRODUCTIVITY': 'Productividad',
 'PARENTING': 'Parenting',
 'PHOTOGRAPHY': '写真'}

In [45]:
grp = apps_df.groupby(['Month_Year', 'Category'])['Installs'].sum().reset_index()
pivot_df = grp.pivot(index='Month_Year', columns='Category', values='Installs')
pivot_df = pivot_df.fillna(0)
pivot_df = pivot_df.sort_index()
pivot_df

Category,PARENTING,PERSONALIZATION,PHOTOGRAPHY,PRODUCTIVITY,TOOLS,TRAVEL_AND_LOCAL
Month_Year,,,,,,
2014-11,0.0,0.0,1000000.0,0.000000e+00,0.0,0.0
2016-10,0.0,0.0,0.0,1.000000e+06,0.0,0.0
2016-12,0.0,2000000.0,0.0,0.000000e+00,0.0,0.0
2017-03,0.0,0.0,50000000.0,0.000000e+00,0.0,0.0
2017-06,0.0,0.0,10000000.0,0.000000e+00,0.0,0.0
2017-07,0.0,0.0,15000000.0,0.000000e+00,0.0,0.0
2017-08,0.0,0.0,0.0,0.000000e+00,50000.0,0.0
2017-09,0.0,1000000.0,0.0,0.000000e+00,0.0,0.0
2017-10,0.0,0.0,5000000.0,0.000000e+00,0.0,50000.0


In [46]:
# cumulative installs over time for each category
cum_df = pivot_df.cumsum()
cum_df

Category,PARENTING,PERSONALIZATION,PHOTOGRAPHY,PRODUCTIVITY,TOOLS,TRAVEL_AND_LOCAL
Month_Year,,,,,,
2014-11,0.0,0.0,1.000000e+06,0.000000e+00,0.0,0.0
2016-10,0.0,0.0,1.000000e+06,1.000000e+06,0.0,0.0
2016-12,0.0,2000000.0,1.000000e+06,1.000000e+06,0.0,0.0
2017-03,0.0,2000000.0,5.100000e+07,1.000000e+06,0.0,0.0
2017-06,0.0,2000000.0,6.100000e+07,1.000000e+06,0.0,0.0
2017-07,0.0,2000000.0,7.600000e+07,1.000000e+06,0.0,0.0
2017-08,0.0,2000000.0,7.600000e+07,1.000000e+06,50000.0,0.0
2017-09,0.0,3000000.0,7.600000e+07,1.000000e+06,50000.0,0.0
2017-10,0.0,3000000.0,8.100000e+07,1.000000e+06,50000.0,50000.0


In [47]:
# month-over-month growth per category, based on the non-cumulative monthly totals
growth_df = pivot_df.pct_change() * 100
growth_df = growth_df.replace([np.inf, -np.inf], np.nan)
growth_df

Category,PARENTING,PERSONALIZATION,PHOTOGRAPHY,PRODUCTIVITY,TOOLS,TRAVEL_AND_LOCAL
Month_Year,,,,,,
2014-11,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,NaN,NaN,-100.000000,NaN,NaN,NaN
2016-12,NaN,NaN,NaN,-100.000000,NaN,NaN
2017-03,NaN,-100.000000,NaN,NaN,NaN,NaN
2017-06,NaN,NaN,-80.000000,NaN,NaN,NaN
2017-07,NaN,NaN,50.000000,NaN,NaN,NaN
2017-08,NaN,NaN,-100.000000,NaN,NaN,NaN
2017-09,NaN,NaN,NaN,NaN,-100.000000,NaN
2017-10,NaN,-100.000000,NaN,NaN,NaN,NaN


In [48]:
ist = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)

current_decimal = now_ist.hour + now_ist.minute / 60
window_start = 16.0   # 4:00 PM
window_end = 18.0     # 6:00 PM

in_window = window_start <= current_decimal <= window_end
print(f"Current IST: {now_ist.strftime('%I:%M %p')} | Chart visible: {in_window}")

Current IST: 01:24 AM | Chart visible: False


In [49]:
colors = {
    'PHOTOGRAPHY': '#1f77b4',
    'TRAVEL_AND_LOCAL': '#ff7f0e',
    'TOOLS': '#2ca02c',
    'PERSONALIZATION': '#d62728',
    'PRODUCTIVITY': '#9467bd',
    'PARENTING': '#8c564b'
}

In [50]:
# chart_base64 holds the chart image so it can be embedded in the dashboard later
chart_base64 = None

if not in_window:
    print("Stacked Area Chart can only be viewed between 4 PM IST and 6 PM IST.")
elif apps_df.empty:
    print("No data available for the selected filtering criteria.")
else:
    cats = list(pivot_df.columns)
    x_pos = list(range(len(pivot_df.index)))
    x_labels = [p.strftime('%b-%Y') for p in pivot_df.index]

    # build the lists stackplot needs, one per category
    stack_data = []
    labels_list = []
    colors_list = []
    for cat in cats:
        stack_data.append(cum_df[cat])
        labels_list.append(legend_names[cat])
        colors_list.append(colors[cat])

    plt.figure(figsize=(14, 7))
    plt.stackplot(x_pos, stack_data, labels=labels_list, colors=colors_list, alpha=0.75)

    # top and bottom edge of each category's band, for drawing the highlight on top
    stack_top = cum_df.cumsum(axis=1)
    stack_bottom = stack_top - cum_df

    # redraw any segment with more than 25% month-over-month growth at full color
    for cat in cats:
        for i in range(1, len(pivot_df)):
            growth = growth_df[cat].iloc[i]
            if pd.notna(growth) and growth > 25:
                xs = [i - 1, i]
                ys_bottom = [stack_bottom[cat].iloc[i - 1], stack_bottom[cat].iloc[i]]
                ys_top = [stack_top[cat].iloc[i - 1], stack_top[cat].iloc[i]]
                plt.fill_between(xs, ys_bottom, ys_top, color=colors[cat])

    plt.legend(title='Category', loc='upper left')
    plt.xticks(x_pos, x_labels, rotation=45, ha='right')
    plt.xlabel('Month-Year')
    plt.ylabel('Cumulative Installs')
    plt.title('Cumulative Installs Over Time by Category (solid band = >25% MoM growth)')
    plt.grid(alpha=0.3)
    plt.tight_layout()

    # save the chart as base64 text so it can be embedded straight into the dashboard html
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    chart_base64 = base64.b64encode(buf.read()).decode('utf-8')

    plt.show()

Stacked Area Chart can only be viewed between 4 PM IST and 6 PM IST.


In [51]:
# build the html body depending on which case we are in
if chart_base64 is not None:
    body = f'<img src="data:image/png;base64,{chart_base64}" style="max-width:100%;">'
elif apps_df.empty:
    body = '<p class="notice">No data available for the selected filtering criteria.</p>'
else:
    body = f'<p class="notice">Stacked Area Chart can only be viewed between 4 PM IST and 6 PM IST.<br>Current IST time: {now_ist.strftime("%I:%M %p")}</p>'

dashboard_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Play Store Dashboard</title>
<style>
body {{ background: #0d0d0d; color: white; font-family: Arial, sans-serif; text-align: center; padding: 30px; }}
h1 {{ color: #a78bfa; margin-bottom: 5px; }}
.subtitle {{ color: #9ca3af; font-size: 14px; margin-bottom: 30px; }}
.notice {{ color: #f59e0b; font-size: 16px; padding: 40px; }}
</style>
</head>
<body>
<h1>Play Store Analytics Dashboard</h1>
<div class="subtitle">Cumulative Installs Over Time by Category | Rating &gt;= 4.2 | Reviews &gt; 1000 | Size 20-80 MB</div>
{body}
</body>
</html>
"""

with open('dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print("Dashboard saved as dashboard.html")
HTML(dashboard_html)

Dashboard saved as dashboard.html
